<a href="https://colab.research.google.com/github/sahmedshereen-prog/Customer-Churn-Prediction/blob/main/Task_1_Uneeq_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Customer Churn Prediction
==========================
Task: Develop a model to predict customer churn for a subscription-based service.

Dataset files (provided separately):
- customer_churn_dataset-training-master.csv
- customer_churn_dataset-testing-master.csv

Columns:
CustomerID, Age, Gender, Tenure, Usage Frequency, Support Calls, Payment Delay,
Subscription Type, Contract Length, Total Spend, Last Interaction, Churn

Churn is already encoded as 0 (stayed) / 1 (churned).

What this script does:
1. Loads the training and testing CSV files separately
2. Cleans and explores the data
3. Encodes categorical features
4. Handles class imbalance (if any) using SMOTE on the training set only
5. Trains multiple classification algorithms
6. Evaluates on the separate test set using precision, recall, F1-score,
   and ROC-AUC (not just accuracy, since accuracy can be misleading on
   imbalanced data)
7. Saves the best model
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    RocCurveDisplay, PrecisionRecallDisplay
)

from imblearn.over_sampling import SMOTE

import joblib

TRAIN_PATH = "customer_churn_dataset-training-master.csv"
TEST_PATH = "customer_churn_dataset-testing-master.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print(train_df.head())

def clean(df):
    df = df.copy()
    df.drop(columns=["CustomerID"], inplace=True)
    df.dropna(inplace=True)
    return df

train_df = clean(train_df)
test_df = clean(test_df)

print("\nClass balance in training set:")
print(train_df["Churn"].value_counts(normalize=True))

print("\nClass balance in testing set:")
print(test_df["Churn"].value_counts(normalize=True))

plt.figure(figsize=(5, 4))
sns.countplot(x="Churn", data=train_df)
plt.title("Churn Distribution in Training Set (0 = Stayed, 1 = Churned)")
plt.savefig("churn_distribution.png", bbox_inches="tight")
plt.close()

categorical_cols = ["Gender", "Subscription Type", "Contract Length"]

encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col])

    test_df = test_df[test_df[col].isin(le.classes_)]
    test_df[col] = le.transform(test_df[col])
    encoders[col] = le

X_train = train_df.drop(columns=["Churn"])
y_train = train_df["Churn"]

X_test = test_df.drop(columns=["Churn"])
y_test = test_df["Churn"]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

print("\nClass balance after SMOTE (train set):")
print(pd.Series(y_train_res).value_counts(normalize=True))

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "SVM (RBF)": SVC(probability=True, random_state=42),
}

results = {}

for name, model in models.items():
    model.fit(X_train_res, y_train_res)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]

    report = classification_report(y_test, y_pred, output_dict=True)
    auc = roc_auc_score(y_test, y_proba)

    results[name] = {
        "precision_churn": report["1"]["precision"],
        "recall_churn": report["1"]["recall"],
        "f1_churn": report["1"]["f1-score"],
        "accuracy": report["accuracy"],
        "roc_auc": auc,
    }

    print(f"\n=== {name} ===")
    print(classification_report(y_test, y_pred, target_names=["No Churn", "Churn"]))
    print("ROC-AUC:", round(auc, 4))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

results_df = pd.DataFrame(results).T.sort_values("recall_churn", ascending=False)
print("\n=== Model comparison (sorted by recall on churn class) ===")
print(results_df)

results_df.to_csv("model_comparison_results.csv")

best_model_name = results_df.index[0]
best_model = models[best_model_name]
print(f"\nBest model based on recall for churners: {best_model_name}")

joblib.dump(best_model, "best_churn_model.pkl")
joblib.dump(scaler, "scaler.pkl")

y_proba_best = best_model.predict_proba(X_test_scaled)[:, 1]

RocCurveDisplay.from_predictions(y_test, y_proba_best)
plt.title(f"ROC Curve - {best_model_name}")
plt.savefig("roc_curve.png", bbox_inches="tight")
plt.close()

PrecisionRecallDisplay.from_predictions(y_test, y_proba_best)
plt.title(f"Precision-Recall Curve - {best_model_name}")
plt.savefig("precision_recall_curve.png", bbox_inches="tight")
plt.close()

print("\nDone. Outputs saved: best_churn_model.pkl, scaler.pkl, "
      "model_comparison_results.csv, churn_distribution.png, "
      "roc_curve.png, precision_recall_curve.png")

Train shape: (318423, 12)
Test shape: (64374, 12)
   CustomerID   Age  Gender  Tenure  Usage Frequency  Support Calls  \
0         2.0  30.0  Female    39.0             14.0            5.0   
1         3.0  65.0  Female    49.0              1.0           10.0   
2         4.0  55.0  Female    14.0              4.0            6.0   
3         5.0  58.0    Male    38.0             21.0            7.0   
4         6.0  23.0    Male    32.0             20.0            5.0   

   Payment Delay Subscription Type Contract Length  Total Spend  \
0           18.0          Standard          Annual        932.0   
1            8.0             Basic         Monthly        557.0   
2           18.0             Basic       Quarterly        185.0   
3            7.0          Standard         Monthly        396.0   
4            8.0             Basic         Monthly        617.0   

   Last Interaction  Churn  
0              17.0    1.0  
1               6.0    1.0  
2               3.0    1.0  
3   